# The Causal Framework: Potential Outcomes, Assumptions, and DAGs

*Companion notebook — Quasi-Experimental Methods for Business Interventions in Python. Self-contained: run the setup cells first.*

In [ ]:
# Install dependencies (Colab/Jupyter)
%pip install -q numpy pandas matplotlib seaborn statsmodels linearmodels scikit-learn scipy doubleml econml dowhy tfp-causalimpact pysyncon rdrobust

## Setup: shared imports and data generators

This cell reproduces the book's `_common.py` so the notebook runs standalone.

In [5]:
# Shared setup for all chapters:
# - Imports standard libraries for data analysis, plotting, and statistical modeling (numpy, pandas, matplotlib, seaborn, statsmodels).
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Set up a random number generator for reproducibility.
rng = np.random.default_rng(42)

# Configure plotting style/theme and pandas display precision.
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# Data generator: creates a synthetic panel dataset simulating store-week revenue
def make_panel(
    n_stores=120,
    n_markets=12,
    n_weeks=104,                # simulate 2 years of weekly data
    treat_markets=(0, 1, 2, 3), # list of market indices to receive treatment
    treat_week=60,              # week when treatment begins
    true_effect=8.0,            # true causal effect (increase in revenue) per week after treatment
    effect_ramp=6,              # number of weeks for the effect to reach its full size (ramps up)
    seed=42,
):
    """
    Generates a synthetic panel (store-level, week-by-week) with a known treatment effect.
    - Each store belongs to a market; only some markets are treated.
    - Simulates realistic variability: store and market fixed effects, seasonal patterns, trends, and random shocks.
    - Outcome variable is revenue.
    - Returns a DataFrame with columns indicating store/market/week, treatment status, and revenue.
    """
    rng = np.random.default_rng(seed)
    stores = np.arange(n_stores)
    markets = stores % n_markets                   # assign stores to markets in round-robin fashion
    store_fe = rng.normal(100, 15, n_stores)       # store-level fixed effects
    market_fe = rng.normal(0, 8, n_markets)        # market-level fixed effects
    store_trend = rng.normal(0.05, 0.03, n_stores) # store-specific growth trends

    rows = []
    market_shock = rng.normal(0, 3, (n_markets, n_weeks)) # week-to-week market-level shocks

    for w in range(n_weeks):
        season = 6 * np.sin(2 * np.pi * w / 52)          # yearly seasonality
        for s in stores:
            m = markets[s]
            treated_unit = m in treat_markets            # is this store's market treated?
            post = w >= treat_week                      # is this after the treatment week?
            # Treatment effect ramps in over effect_ramp weeks after the launch
            if treated_unit and post:
                ramp = min(1.0, (w - treat_week + 1) / effect_ramp)
                tau = true_effect * ramp
            else:
                tau = 0.0
            # Revenue: sum of all deterministic + stochastic elements
            y = (store_fe[s] + market_fe[m] + store_trend[s] * w
                 + season + market_shock[m, w]
                 + tau + rng.normal(0, 4))              # idiosyncratic noise
            # Columns: store, market, week, treated, post, D (treated & post), revenue
            rows.append((s, m, w, int(treated_unit), int(post),
                         int(treated_unit and post), y))

    return pd.DataFrame(rows, columns=[
        "store", "market", "week", "treated", "post", "D", "revenue"])

# Data generator: creates a synthetic customer-level dataset for a loyalty program
def make_customers(n=8000, true_att=12.0, seed=42):
    """
    Generates customer-level data for a loyalty program scenario where treatment is adopted non-randomly.
    - Covariates: tenure, baseline spend, engagement, urban/rural.
    - Simulates treatment selection based on covariates (selection on observables).
    - Outcome y depends on covariates and treatment.
    - Returns a DataFrame with features + treatment indicator + outcome.
    """
    rng = np.random.default_rng(seed)
    tenure = rng.gamma(3, 1.2, n)                            # customer tenure (in years)
    spend0 = rng.lognormal(4.0, 0.5, n)                      # baseline spend
    engage = rng.beta(2, 5, n) * 10                          # engagement index
    urban = rng.binomial(1, 0.55, n)                         # urban location indicator
    # Propensity to adopt treatment is a logistic function of covariates
    logit = -3.0 + 0.25*tenure + 0.004*spend0 + 0.30*engage + 0.4*urban
    p = 1 / (1 + np.exp(-logit))
    D = rng.binomial(1, p)                                   # treatment assignment
    y = (0.55*spend0 + 4*tenure + 2.5*engage + 6*urban
         + true_att * D + rng.normal(0, 12, n))              # outcome: function of X + treatment + noise
    return pd.DataFrame(dict(tenure=tenure, spend0=spend0, engage=engage,
                             urban=urban, D=D, y=y))

# Generate synthetic datasets on import:
panel = make_panel()                  # Simulated store-week panel dataset
cust = make_customers()               # Simulated customer-level dataset
X_cols = ["tenure", "spend0", "engage", "urban"]  # Feature columns for customer models
TRUE_ATT = 12.0                       # "True" treatment effect in the customer data (for simulation/benchmarking)


## Potential outcomes and the fundamental problem

For each unit *i* (store, customer, market) define two **potential outcomes**:

- $Y_i(1)$ — revenue if the store receives the program,
- $Y_i(0)$ — revenue if it does not.

The individual causal effect is $\tau_i = Y_i(1) - Y_i(0)$. The **fundamental problem of causal inference** is that we only ever observe one of the two: the *factual* outcome. The other is the **counterfactual**. Every method in this tutorial is a strategy for estimating the missing counterfactual.

Common estimands:

| Estimand | Definition | Business question |
|---|---|---|
| **ATE** | $E[Y(1) - Y(0)]$ | Average effect if *everyone* were treated |
| **ATT** | $E[Y(1) - Y(0) \mid D=1]$ | Effect on the units that actually got the program (most common in rollout evaluations) |
| **LATE** | Effect on *compliers* | Effect among those whose treatment status was moved by an instrument (Section 11) |
| **CATE** | $E[Y(1) - Y(0) \mid X=x]$ | Effect for a specific segment — fuel for targeting (Section 12) |

Being explicit about the estimand matters: a synthetic control estimates the ATT for one treated market; an IV estimates a LATE that may differ sharply from the ATE. Reporting "the effect of the program" without saying *for whom* is a common source of stakeholder confusion.

## Why naive comparisons fail: selection bias

The naive estimator compares treated and untreated outcomes:

$$
\underbrace{E[Y\mid D=1] - E[Y\mid D=0]}_{\text{naive difference}}
= \underbrace{E[Y(1)-Y(0)\mid D=1]}_{\text{ATT}}
+ \underbrace{E[Y(0)\mid D=1] - E[Y(0)\mid D=0]}_{\text{selection bias}}
$$

In business data, selection bias is the rule, not the exception: programs launch first in flagship markets; loyalty programs attract the already-loyal; struggling stores get the turnaround intervention (producing *negative* bias and the appearance that help hurts — a business version of "ashtrays cause cancer").

In [7]:
# Naive post-treatment comparison: difference in mean revenue between treated and control markets
# 1. Filter the panel data to only include weeks after the treatment started (week >= 60)
post_df = panel.query("week >= 60")

# 2. Calculate the mean revenue for the treated markets in the post-period
mean_treated = post_df.query("treated == 1").revenue.mean()

# 3. Calculate the mean revenue for the control (untreated) markets in the post-period
mean_control = post_df.query("treated == 0").revenue.mean()

# 4. Compute the naive difference: treated - control (this mixes treatment effect and pre-existing differences)
naive = mean_treated - mean_control

# 5. Print the result along with a reminder of the true effect for reference
print(f"Naive post-period difference: {naive:.2f}  (true effect ≈ 8)")

Naive post-period difference: 9.47  (true effect ≈ 8)


In our simulation, market fixed effects make treated and control markets differ at baseline, so the naive estimate mixes the true effect with pre-existing level differences.

## The identifying assumptions, method by method

Every quasi-experimental method buys identification with an assumption:

| Method | Identifying assumption (informal) |
|---|---|
| DiD | Treated and control groups would have moved in **parallel** absent treatment |
| ITS / CausalImpact | The pre-period model (trend/seasonality/covariates) would have **continued** absent treatment |
| Synthetic control | A weighted combo of donors reproduces the treated unit's counterfactual path |
| Propensity methods | **No unmeasured confounders** (conditional ignorability) + overlap |
| RDD | Units just above/below a threshold are **as good as randomized** |
| IV | Instrument affects outcome **only through** treatment (exclusion) |
| DML | Same as propensity methods, but nuisances estimated flexibly with ML |

None of these can be proven from data. The craft is choosing the assumption that best matches the *known assignment mechanism* and then attacking it with falsification tests (Section 13).

## DAGs: confounders, mediators, and colliders

A directed acyclic graph (DAG) encodes your assumptions about what causes what, and tells you what to adjust for.

```
        Market affluence (W)
          /            \
         v              v
   Program (D)  ---->  Revenue (Y)
```

- **Confounder (W):** causes both D and Y → must adjust.
- **Mediator (M):** D → M → Y (e.g., the program increases foot traffic which increases revenue). Adjusting for a mediator *removes* part of the effect you want to measure. Do not control for post-treatment variables.
- **Collider (C):** D → C ← Y. Adjusting for a collider *creates* bias. Classic business example: conditioning on "stores that stayed open" when the program affects closure.

`dowhy` lets you encode the DAG and machine-check identification:

In [10]:
# Import dowhy and CausalModel for causal inference based on DAGs
import dowhy
from dowhy import CausalModel

# Step 1: Aggregate post-treatment outcomes for each store.
# - Only keep data from week 60 and beyond (the post-program period).
# - For each store, calculate:
#     - Average revenue after the program (revenue)
#     - Whether it was treated (max of treated)
#     - Its market assignment (first value of market)
cross = (
    panel.query("week >= 60")
         .groupby("store")
         .agg(
             revenue=("revenue", "mean"),
             treated=("treated", "max"),
             market=("market", "first")
         )
         .reset_index()
)

# Step 2: Simulate a confounder "market affluence"
# - For each market, take the mean post-program revenue as its "affluence"
# - Affluence is assigned to each store in the same market plus small random noise per store
affluence = cross.groupby("market").revenue.transform("mean")
cross["affluence"] = affluence + rng.normal(0, 1, len(cross))

# Step 3: Specify the causal model based on a DAG:
# - Outcome: revenue
# - Treatment: treated
# - Common cause (confounder): affluence
model = CausalModel(
    data=cross,
    treatment="treated",
    outcome="revenue",
    common_causes=["affluence"],
)

# Step 4: Check if the effect of D on Y is identified given our DAG and print the result.
identified = model.identify_effect(proceed_when_unidentifiable=True)
print(identified)

Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
    d                           
──────────(E[revenue|affluence])
d[treated]                      
Estimand assumption 1, Unconfoundedness: If U→{treated} and U→revenue then P(revenue|treated,affluence,U) = P(revenue|treated,affluence)

### Estimand : 2
Estimand name: iv
No such variable(s) found!

### Estimand : 3
Estimand name: frontdoor
No such variable(s) found!

### Estimand : 4
Estimand name: general_adjustment
Estimand expression:
    d                           
──────────(E[revenue|affluence])
d[treated]                      
Estimand assumption 1, Unconfoundedness: If U→{treated} and U→revenue then P(revenue|treated,affluence,U) = P(revenue|treated,affluence)



The rest of the tutorial is organized by *assignment mechanism*: assignment by group-and-time (Sections 4–8), by observables (Section 9, 12), by threshold (Section 10), and by instrument (Section 11).